# QLoRA Fine-Tuning & CPU Inference — Qwen2.5-1.5B-Instruct

End-to-end notebook: 4-bit QLoRA fine-tuning on GPU -> merge adapter -> CPU inference -> evaluation.

**How to use this notebook:**
1. Run the **Setup** cell, then **Runtime -> Restart session**, then continue from the **Upload Dataset** cell (no need to re-run Setup).
2. Upload `train.csv` when prompted by the **Upload Dataset** cell.
3. Run **Data Prep**, **QLoRA Fine-Tuning**, and **Merge Adapter** on a GPU runtime (Runtime -> Change runtime type -> GPU).
4. Download `test_split.jsonl` and `qwen_qlora_merged.zip` using the download cells provided (unzip the latter locally).
5. Run **CPU Inference Comparison** and **Evaluation** either by switching this Colab runtime to CPU, or locally in VS Code with the downloaded files.

## 1. Setup
Install dependencies. bitsandbytes/peft/accelerate are needed for the GPU (QLoRA) part; rouge-score/sacrebleu are only needed for the evaluation part at the end.

**Important (Colab):** run the install cell below, then **Runtime -> Restart session**, then continue from the Upload Dataset cell onward (no need to re-run the install cell). This avoids a common `torchvision::nms` / `ModuleNotFoundError` crash caused by torch and torchvision versions getting out of sync after installing packages — we don't use torchvision anywhere in this notebook (text-only model), so we remove it instead of trying to match versions.

In [ ]:
!pip uninstall -y torchvision -q
!pip install -q -U transformers accelerate peft bitsandbytes datasets psutil pandas scikit-learn rouge-score sacrebleu

print("Install done. Now go to Runtime -> Restart session, then run the cells below (skip this cell on re-run).")


## 2. Upload Dataset
Uploads `train.csv` (Alpaca-style `instruction / input / output` CSV) into the Colab working directory. Run this cell and pick the file from your computer when the upload widget appears.

In [ ]:
from google.colab import files

uploaded = files.upload()  # select train.csv in the dialog
print("Uploaded:", list(uploaded.keys()))


## 3. Data Prep
data_prep.py
Splits train.csv (Alpaca-style instruction/input/output dataset) into a
train set (used for QLoRA fine-tuning) and a held-out test set (used later
for CPU inference + evaluation, comparing base vs fine-tuned model).

Run this once, locally or in Colab. Only needs pandas + scikit-learn.
    pip install pandas scikit-learn

In [ ]:
import json
import pandas as pd
from sklearn.model_selection import train_test_split

# ---------------- CONFIG ----------------
CSV_PATH = "train.csv"

# Free-tier GPUs (Colab T4 / Kaggle P100) have limited time+VRAM, and this
# task is about studying QLoRA, not squeezing max accuracy. So we
# deliberately fine-tune on a small subset rather than all 51,760 rows.
TRAIN_SAMPLE_SIZE = 3000     # rows used to fine-tune
TEST_SAMPLE_SIZE = 200       # rows held out for evaluation/inference
RANDOM_SEED = 42

TRAIN_OUT = "train_split.jsonl"
TEST_OUT = "test_split.jsonl"

ALPACA_PROMPT = """Below is an instruction that describes a task{input_clause}.
Write a response that appropriately completes the request.

### Instruction:
{instruction}
{input_block}
### Response:
"""


def build_prompt(instruction: str, input_text: str) -> str:
    has_input = isinstance(input_text, str) and input_text.strip() != ""
    input_clause = ", paired with an input that provides further context" if has_input else ""
    input_block = f"\n### Input:\n{input_text}\n" if has_input else ""
    return ALPACA_PROMPT.format(
        input_clause=input_clause, instruction=instruction, input_block=input_block
    )


def main():
    df = pd.read_csv(CSV_PATH)
    df["input"] = df["input"].fillna("")

    # Shuffle + split. We take a stratified-by-nothing random split since
    # this is an open-ended instruction dataset (no class labels).
    df = df.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)

    train_pool, test_pool = train_test_split(
        df, test_size=0.1, random_state=RANDOM_SEED
    )

    train_df = train_pool.sample(
        n=min(TRAIN_SAMPLE_SIZE, len(train_pool)), random_state=RANDOM_SEED
    )
    test_df = test_pool.sample(
        n=min(TEST_SAMPLE_SIZE, len(test_pool)), random_state=RANDOM_SEED
    )

    def write_jsonl(frame, path, for_training):
        with open(path, "w", encoding="utf-8") as f:
            for _, row in frame.iterrows():
                prompt = build_prompt(row["instruction"], row["input"])
                record = {
                    "instruction": row["instruction"],
                    "input": row["input"],
                    "output": row["output"],
                    "prompt": prompt,
                }
                if for_training:
                    # full text = prompt + reference answer + eos marker text
                    record["text"] = prompt + row["output"]
                f.write(json.dumps(record, ensure_ascii=False) + "\n")

    write_jsonl(train_df, TRAIN_OUT, for_training=True)
    write_jsonl(test_df, TEST_OUT, for_training=False)

    print(f"Full dataset: {len(df)} rows")
    print(f"Train split written to {TRAIN_OUT}: {len(train_df)} rows")
    print(f"Test split written to {TEST_OUT}: {len(test_df)} rows")


if __name__ == "__main__":
    main()

main()


### Download the test split
You'll need `test_split.jsonl` later on your CPU machine (VS Code) for the inference comparison and evaluation steps.

In [ ]:
from google.colab import files
files.download("test_split.jsonl")


## 4. QLoRA Fine-Tuning (run on GPU)
train_qlora.py
QLoRA fine-tuning of Qwen2.5-1.5B-Instruct on the Alpaca-style dataset
prepared by data_prep.py. Meant to be run on a free-tier GPU (Colab T4 /
Kaggle P100/T4).

    pip install -U transformers accelerate peft bitsandbytes datasets torch

Workflow:
    1. Load base model in 4-bit (NF4, double-quantized) -> frozen weights
    2. Attach LoRA adapters on the attention projections
    3. Train only the adapters
    4. Save the adapter (a few MB) to ADAPTER_DIR

Set USE_4BIT_QUANTIZATION = False to instead run plain LoRA (fp16 base,
no quantization) on the same data/hyperparameters, so you can compare GPU
memory usage and training speed against QLoRA (see README for how to log
this comparison).

In [ ]:
import json
import time

import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

# ---------------- CONFIG ----------------
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
TRAIN_FILE = "train_split.jsonl"
ADAPTER_DIR = "./qwen_qlora_adapter"
MAX_SEQ_LEN = 512

# Toggle this to compare QLoRA (True) vs standard LoRA (False) on the
# same setup, for the "LoRA vs QLoRA" trade-off writeup.
USE_4BIT_QUANTIZATION = True

# ---- Quantization config (only used if USE_4BIT_QUANTIZATION) ----
# nf4: 4-bit NormalFloat, a data type designed for normally-distributed
#      weights -- gives better accuracy than plain int4 at the same size.
# double_quant: quantizes the quantization constants themselves, saving
#      ~0.4 bits/parameter more, at negligible extra compute cost.
# compute_dtype: matmuls are de-quantized to this dtype on the fly during
#      the forward/backward pass. bf16 is used since it's what the base
#      model was trained/released in and avoids overflow issues fp16 can have.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# ---- LoRA config ----
# rank (r): dimensionality of the low-rank update matrices. Higher r =
#      more trainable capacity but more memory/compute and higher overfit
#      risk on a small dataset. r=16 is a common middle ground for a 1.5B
#      model fine-tuned on a few thousand examples.
# alpha: scales the LoRA update (effective scale = alpha / r). alpha=32
#      with r=16 gives a scale of 2, a widely-used default that keeps the
#      adapter's contribution meaningful without overpowering the frozen
#      base weights.
# dropout: regularizes the adapter since we're training on a fairly small
#      sample; 0.05 is a light amount, enough to reduce overfitting
#      without slowing convergence much.
# target_modules: q_proj/k_proj/v_proj/o_proj are the four attention
#      projections. Adapting all four (rather than just q_proj/v_proj)
#      gives the adapter more expressive control over attention at a
#      modest extra parameter cost (still <1% of total params), which
#      matters here because we only have a few thousand training examples
#      and want the adapter to actually shift behaviour.
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

# ---- Training hyperparameters ----
# learning_rate: 2e-4 is standard for LoRA/QLoRA -- much higher than a
#      full fine-tune LR (~1e-5) because only a small number of new
#      parameters are being trained from scratch.
# epochs: 3 passes over ~3k examples is enough to see a clear behavioural
#      shift without long training time on a free-tier GPU.
# batch_size / grad_accum: a small per-device batch (limited by 16GB
#      T4 VRAM once the optimizer states + activations are counted) with
#      gradient accumulation to reach an effective batch size of 16.
train_args = TrainingArguments(
    output_dir="./qlora_train_output",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,   # effective batch size = 16
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=20,
    save_strategy="no",
    bf16=True,
    optim="paged_adamw_8bit" if USE_4BIT_QUANTIZATION else "adamw_torch",
    report_to="none",
)


def load_model_and_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if USE_4BIT_QUANTIZATION:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            quantization_config=bnb_config,
            device_map="auto",
        )
        model = prepare_model_for_kbit_training(model)
    else:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.bfloat16,
            device_map="auto",
        )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model, tokenizer


def tokenize_dataset(dataset, tokenizer):
    def _tokenize(example):
        tokens = tokenizer(
            example["text"],
            truncation=True,
            max_length=MAX_SEQ_LEN,
            padding="max_length",
        )
        tokens["labels"] = tokens["input_ids"].copy()
        return tokens

    return dataset.map(_tokenize, remove_columns=dataset.column_names)


def main():
    model, tokenizer = load_model_and_tokenizer()

    raw_dataset = load_dataset("json", data_files=TRAIN_FILE, split="train")
    tokenized_dataset = tokenize_dataset(raw_dataset, tokenizer)

    collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

    trainer = Trainer(
        model=model,
        args=train_args,
        train_dataset=tokenized_dataset,
        data_collator=collator,
    )

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    start = time.time()
    trainer.train()
    elapsed = time.time() - start

    peak_mem_gb = (
        torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else None
    )

    print(f"\nTraining mode: {'QLoRA (4-bit)' if USE_4BIT_QUANTIZATION else 'LoRA (fp16/bf16)'}")
    print(f"Training time: {elapsed:.1f} sec")
    print(f"Peak GPU memory allocated: {peak_mem_gb:.2f} GB" if peak_mem_gb else "No GPU detected")

    model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    print(f"Adapter saved to {ADAPTER_DIR}")

    # Log a small summary so it can be pasted into the documentation later.
    # summary = {
    #     "mode": "qlora_4bit" if USE_4BIT_QUANTIZATION else "lora_fp16",
    #     "training_time_sec": elapsed,
    #     "peak_gpu_memory_gb": peak_mem_gb,
    #     "lora_r": lora_config.r,
    #     "lora_alpha": lora_config.lora_alpha,
    #     "target_modules": list(lora_config.target_modules),
    #     "learning_rate": train_args.learning_rate,
    #     "epochs": train_args.num_train_epochs,
    #     "effective_batch_size": train_args.per_device_train_batch_size
    #     * train_args.gradient_accumulation_steps,
    # }
    # with open("train_run_summary.json", "a") as f:
    #     f.write(json.dumps(summary) + "\n")


if __name__ == "__main__":
    main()

main()


## 5. Merge LoRA Adapter into Base Model (run on GPU)
merge_lora.py
Step 3 of the workflow: dequantize the base model to full precision (fp16)
and merge the trained LoRA adapter into it, producing a single standalone
model that no longer needs bitsandbytes/peft to run -- suitable for plain
CPU inference afterward.

Run this on the GPU machine (Colab/Kaggle), right after train_qlora.py.
Then download the MERGED_DIR folder and copy it to your VS Code machine.

    pip install -U transformers peft torch

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_DIR = "./qwen_qlora_adapter"     # output of train_qlora.py
MERGED_DIR = "./qwen_qlora_merged"       # final standalone fp16 model


def main():
    print("Loading base model in fp16 (dequantized, full precision)...")
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    print(f"Loading LoRA adapter from {ADAPTER_DIR} and attaching to base model...")
    merged_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)

    print("Merging adapter weights into base weights (merge_and_unload)...")
    # merge_and_unload() folds W_lora = B @ A * (alpha/r) into the base
    # weight matrices (W' = W + W_lora) and then discards the separate
    # adapter matrices -- the result behaves like a normal dense model.
    merged_model = merged_model.merge_and_unload()

    print(f"Saving merged fp16 model to {MERGED_DIR} ...")
    merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
    tokenizer.save_pretrained(MERGED_DIR)

    print("Done. Copy the MERGED_DIR folder to your CPU machine for inference.")


if __name__ == "__main__":
    main()

main()


### Download the merged model
Zips the `qwen_qlora_merged/` folder and downloads it to your computer (Colab file picker popup) so you can copy it to VS Code for CPU inference. The folder is a few GB (fp16 weights), so this may take a minute.

If the browser download is unreliable for you, an alternative is to mount Google Drive and copy the folder there instead — ask if you'd like that version.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("qwen_qlora_merged", "zip", MERGED_DIR)
files.download("qwen_qlora_merged.zip")


## 6. CPU Inference Comparison
inference_compare.py
Step 4 of the workflow: run inference on CPU with (a) the original base
Qwen2.5-1.5B-Instruct model and (b) the merged fine-tuned model, on the
same held-out test prompts. Records loading time, per-prompt generation
latency, and tokens/sec for both, and saves everything to a CSV for
evaluate.py to score.

Run this in VS Code / locally, on CPU (no GPU needed).

    pip install -U transformers torch psutil pandas

**Note:** If you're continuing in the same Colab session, switch the runtime to CPU first (Runtime -> Change runtime type -> None), or just run this section locally in VS Code with the `qwen_qlora_merged/` folder and `test_split.jsonl` copied over.

In [ ]:
import csv
import json
import time

import psutil
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
FINETUNED_MODEL_DIR = "./qwen_qlora_merged"   # output of merge_lora.py
TEST_FILE = "test_split.jsonl"
NUM_TEST_PROMPTS = 20          # keep small -- CPU generation is slow
MAX_NEW_TOKENS = 150
RESULTS_CSV = "inference_results.csv"

torch.set_num_threads(psutil.cpu_count(logical=True))


def current_rss_gb():
    return psutil.Process().memory_info().rss / 1e9


def load_model(path_or_name):
    mem_before = current_rss_gb()
    start = time.time()
    tokenizer = AutoTokenizer.from_pretrained(path_or_name)
    model = AutoModelForCausalLM.from_pretrained(
        path_or_name,
        torch_dtype=torch.float32,   # plain CPU inference, no quantization
    )
    model.eval()
    load_time = time.time() - start
    mem_after = current_rss_gb()
    print(f"Loaded {path_or_name} in {load_time:.1f}s, "
          f"RAM used: {mem_after - mem_before:.2f} GB")
    return model, tokenizer, load_time, mem_after - mem_before


def generate(model, tokenizer, prompt):
    inputs = tokenizer(prompt, return_tensors="pt")
    start = time.time()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,          # greedy, so base vs fine-tuned is a fair comparison
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.time() - start
    new_tokens = output_ids.shape[1] - inputs["input_ids"].shape[1]
    text = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    tokens_per_sec = new_tokens / elapsed if elapsed > 0 else 0.0
    return text, elapsed, new_tokens, tokens_per_sec


def main():
    test_rows = []
    with open(TEST_FILE, "r", encoding="utf-8") as f:
        for line in f:
            test_rows.append(json.loads(line))
    test_rows = test_rows[:NUM_TEST_PROMPTS]

    print("=== Loading base (un-fine-tuned) model ===")
    base_model, base_tok, base_load_time, base_load_mem = load_model(BASE_MODEL_NAME)

    print("=== Loading fine-tuned (merged) model ===")
    ft_model, ft_tok, ft_load_time, ft_load_mem = load_model(FINETUNED_MODEL_DIR)

    rows = []
    for i, row in enumerate(test_rows):
        prompt = row["prompt"]
        reference = row["output"]

        base_text, base_time, base_ntok, base_tps = generate(base_model, base_tok, prompt)
        ft_text, ft_time, ft_ntok, ft_tps = generate(ft_model, ft_tok, prompt)

        print(f"[{i+1}/{len(test_rows)}] base {base_time:.1f}s ({base_tps:.1f} tok/s) | "
              f"finetuned {ft_time:.1f}s ({ft_tps:.1f} tok/s)")

        rows.append({
            "instruction": row["instruction"],
            "input": row["input"],
            "reference_output": reference,
            "base_output": base_text,
            "base_latency_sec": base_time,
            "base_tokens_per_sec": base_tps,
            "finetuned_output": ft_text,
            "finetuned_latency_sec": ft_time,
            "finetuned_tokens_per_sec": ft_tps,
        })

    with open(RESULTS_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)

    print(f"\nSaved per-prompt results to {RESULTS_CSV}")
    print("\n--- Loading summary ---")
    print(f"Base model load time: {base_load_time:.1f}s, RAM: {base_load_mem:.2f} GB")
    print(f"Fine-tuned model load time: {ft_load_time:.1f}s, RAM: {ft_load_mem:.2f} GB")

    with open("loading_summary.json", "w") as f:
        json.dump({
            "base_load_time_sec": base_load_time,
            "base_load_ram_gb": base_load_mem,
            "finetuned_load_time_sec": ft_load_time,
            "finetuned_load_ram_gb": ft_load_mem,
        }, f, indent=2)


if __name__ == "__main__":
    main()

main()


## 7. Evaluation (ROUGE / BLEU / latency)
evaluate.py
Scores the base-vs-fine-tuned outputs produced by inference_compare.py
against the reference (ground-truth) outputs, using standard text
generation metrics (ROUGE-1/2/L, BLEU), plus prints the latency/speed
comparison. This is the "evaluation metrics" deliverable.

    pip install -U rouge-score sacrebleu pandas

In [ ]:
import json

import pandas as pd
from rouge_score import rouge_scorer
import sacrebleu

RESULTS_CSV = "inference_results.csv"
LOADING_SUMMARY = "loading_summary.json"


def score_column(df, output_col, reference_col="reference_output"):
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

    rouge1, rouge2, rougeL, bleu_scores = [], [], [], []
    for ref, hyp in zip(df[reference_col], df[output_col]):
        ref, hyp = str(ref), str(hyp)
        scores = scorer.score(ref, hyp)
        rouge1.append(scores["rouge1"].fmeasure)
        rouge2.append(scores["rouge2"].fmeasure)
        rougeL.append(scores["rougeL"].fmeasure)
        bleu_scores.append(sacrebleu.sentence_bleu(hyp, [ref]).score)

    return {
        "rouge1": sum(rouge1) / len(rouge1),
        "rouge2": sum(rouge2) / len(rouge2),
        "rougeL": sum(rougeL) / len(rougeL),
        "bleu": sum(bleu_scores) / len(bleu_scores),
    }


def main():
    df = pd.read_csv(RESULTS_CSV)

    base_scores = score_column(df, "base_output")
    ft_scores = score_column(df, "finetuned_output")

    print("=== Text quality metrics (vs. reference outputs) ===")
    print(f"{'Metric':<10}{'Base':>10}{'Fine-tuned':>14}")
    for metric in ["rouge1", "rouge2", "rougeL", "bleu"]:
        print(f"{metric:<10}{base_scores[metric]:>10.4f}{ft_scores[metric]:>14.4f}")

    print("\n=== Latency / speed (CPU inference) ===")
    print(f"Base avg latency:       {df['base_latency_sec'].mean():.2f} sec/prompt, "
          f"{df['base_tokens_per_sec'].mean():.2f} tok/s")
    print(f"Fine-tuned avg latency: {df['finetuned_latency_sec'].mean():.2f} sec/prompt, "
          f"{df['finetuned_tokens_per_sec'].mean():.2f} tok/s")

    try:
        with open(LOADING_SUMMARY) as f:
            loading = json.load(f)
        print("\n=== Model loading (CPU) ===")
        print(f"Base:       {loading['base_load_time_sec']:.1f}s, "
              f"{loading['base_load_ram_gb']:.2f} GB RAM")
        print(f"Fine-tuned: {loading['finetuned_load_time_sec']:.1f}s, "
              f"{loading['finetuned_load_ram_gb']:.2f} GB RAM")
    except FileNotFoundError:
        pass

    summary_rows = [
        {"model": "base", **base_scores,
         "avg_latency_sec": df["base_latency_sec"].mean(),
         "avg_tokens_per_sec": df["base_tokens_per_sec"].mean()},
        {"model": "finetuned", **ft_scores,
         "avg_latency_sec": df["finetuned_latency_sec"].mean(),
         "avg_tokens_per_sec": df["finetuned_tokens_per_sec"].mean()},
    ]
    pd.DataFrame(summary_rows).to_csv("evaluation_summary.csv", index=False)
    print("\nSaved evaluation_summary.csv")


if __name__ == "__main__":
    main()

main()
